In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import re

In [ ]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

In [ ]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
# Let's make a conversation between gemma-4-31b-it and gemini-2.5-flash

gemma_model = "gemma-4-31b-it"
gemini_model = "gemini-3.1-flash-lite"

gemma_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

gemini_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gemma_messages = ["Hi"]
gemini_messages = ["Hi there"]

In [ ]:
def clean_response(text):
    return re.sub(
        r"<thought>.*?</thought>",
        "",
        text,
        flags=re.DOTALL
    ).strip()

In [ ]:
def call_gemma():
    messages = [{"role": "system", "content": gemma_system}]
    for gemma, gemini in zip(gemma_messages, gemini_messages):
        messages.append({"role": "assistant", "content": gemma})
        messages.append({"role": "user", "content": gemini})
    response = openai.chat.completions.create(model=gemma_model, messages=messages)
    raw = response.choices[0].message.content
   # Remove reasoning traces from the response, which are enclosed in <thought> tags
    return clean_response(raw)

In [ ]:
#call_gemma()

In [ ]:
def call_gemini():
    messages = [{"role": "system", "content": gemini_system}]
    for gemma, gemini in zip(gemma_messages, gemini_messages):
        messages.append({"role": "user", "content": gemma})
        messages.append({"role": "assistant", "content": gemini})
    messages.append({"role": "user", "content": gemma_messages[-1]})
    response = openai.chat.completions.create(model=gemini_model, messages=messages)
    raw = response.choices[0].message.content
    # Remove reasoning traces from the response, which are enclosed in <thought> tags
    return clean_response(raw)

In [ ]:
#call_gemini()

In [ ]:
#call_gemma()

In [ ]:
gemma_messages = ["Hi"]
gemini_messages = ["Hi there"]

display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))
display(Markdown(f"### Gemma:\n{gemma_messages[0]}\n"))

for i in range(2):
    gemma_next = call_gemma()
    display(Markdown(f"### Gemma:\n{gemma_next}\n"))
    gemma_messages.append(gemma_next)

    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)
